# Algorithms 01: Introductory Coding**Learning Objectives:**1. Review basic control flow (loops, if/else) without using external libraries2. Solve common algorithmic primitives from scratch3. Learn basic testing methodologies4. Develop strong debugging habits using print statements**Prerequisites:** Basic programming familiarity**Estimated time:** 120 minutes**Textbook:** *Justin Skycak — Intro to Algorithms & Machine Learning* Chapter 1

In [ ]:
# Google Colab Setup!pip install -q ipywidgets jupyterquiz ipytest plotly sympy networkx pandas numpy matplotlib tqdmimport json, os, sysfrom pathlib import Pathif Path('/content').exists():    repo_root = Path('/content/upskill')else:    repo_root = Path.cwd()if str(repo_root) not in sys.path:    sys.path.insert(0, str(repo_root))try:    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )except ModuleNotFoundError:    Path('learning_tools.py').write_text('"""Colab-first learning helpers for the interactive math curriculum.\n\nThe functions in this module are intentionally lightweight: they work in a\nplain notebook, in Google Colab, and in local Jupyter without requiring a\nbackend service.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict, field\nfrom datetime import datetime, timedelta, timezone\nimport json\nimport math\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\n\nSCHEMA_VERSION = 1\nDEFAULT_PROGRESS_FILENAME = "upskill_math_progress.json"\nDRIVE_PROGRESS_DIR = "MyDrive/upskill"\n\n\ndef _utcnow() -> datetime:\n    return datetime.now(timezone.utc)\n\n\ndef _iso(dt: datetime) -> str:\n    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat()\n\n\ndef _parse_iso(value: str | None) -> datetime | None:\n    if not value:\n        return None\n    try:\n        return datetime.fromisoformat(value.replace("Z", "+00:00"))\n    except ValueError:\n        return None\n\n\n@dataclass\nclass Skill:\n    """A small curriculum skill descriptor."""\n\n    id: str\n    title: str\n    notebook: str\n    level: int\n    prerequisites: list[str] = field(default_factory=list)\n    tags: list[str] = field(default_factory=list)\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef setup_colab(\n    progress_filename: str = DEFAULT_PROGRESS_FILENAME,\n    mount_drive: bool = True,\n    verbose: bool = True,\n) -> Path:\n    """Prepare a notebook session and return the progress-file path.\n\n    In Colab, this mounts Google Drive when available. Outside Colab, it falls\n    back to the current working directory.\n    """\n\n    progress_path = Path(progress_filename)\n    try:\n        import google.colab  # type: ignore  # noqa: F401\n        in_colab = True\n    except Exception:\n        in_colab = False\n\n    if in_colab and mount_drive:\n        try:\n            from google.colab import drive  # type: ignore\n\n            drive.mount("/content/drive")\n            drive_dir = Path("/content/drive") / DRIVE_PROGRESS_DIR\n            drive_dir.mkdir(parents=True, exist_ok=True)\n            progress_path = drive_dir / progress_filename\n        except Exception as exc:\n            if verbose:\n                print(f"Drive mount skipped; using local progress file. ({exc})")\n\n    if verbose:\n        print(f"Learning environment ready. Progress: {progress_path}")\n    return progress_path\n\n\nclass ProgressStore:\n    """JSON-backed spaced-repetition progress store."""\n\n    def __init__(self, path: str | Path | None = None, learner_id: str = "local"):\n        self.path = Path(path or DEFAULT_PROGRESS_FILENAME)\n        self.learner_id = learner_id\n        self.data = self._load()\n\n    def _default_data(self) -> dict[str, Any]:\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "learner_id": self.learner_id,\n            "skills": {},\n        }\n\n    def _load(self) -> dict[str, Any]:\n        if not self.path.exists():\n            return self._default_data()\n        try:\n            data = json.loads(self.path.read_text(encoding="utf-8"))\n        except Exception:\n            return self._default_data()\n        if data.get("schema_version") != SCHEMA_VERSION:\n            data["schema_version"] = SCHEMA_VERSION\n        data.setdefault("learner_id", self.learner_id)\n        data.setdefault("skills", {})\n        return data\n\n    def save(self) -> None:\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.path.write_text(json.dumps(self.data, indent=2), encoding="utf-8")\n\n    def get(self, skill_id: str) -> dict[str, Any]:\n        return self.data.setdefault("skills", {}).setdefault(skill_id, {\n            "attempts": 0,\n            "correct": 0,\n            "confidence": 0,\n            "mastery": 0.0,\n            "last_seen": None,\n            "next_review": None,\n        })\n\n    def record_attempt(\n        self,\n        skill_id: str,\n        correct: bool,\n        confidence: int = 3,\n        item_type: str = "practice",\n    ) -> dict[str, Any]:\n        confidence = max(1, min(5, int(confidence)))\n        row = self.get(skill_id)\n        attempts = int(row.get("attempts", 0)) + 1\n        correct_count = int(row.get("correct", 0)) + int(bool(correct))\n        accuracy = correct_count / attempts\n        confidence_factor = confidence / 5\n        mastery = round((0.75 * accuracy) + (0.25 * confidence_factor), 3)\n\n        row.update({\n            "attempts": attempts,\n            "correct": correct_count,\n            "confidence": confidence,\n            "mastery": mastery,\n            "last_seen": _iso(_utcnow()),\n            "last_item_type": item_type,\n            "next_review": _iso(_utcnow() + review_interval(correct, confidence, attempts)),\n        })\n        self.save()\n        return row\n\n    def review_queue(\n        self,\n        skills: Iterable[Skill | dict[str, Any]] | None = None,\n        limit: int = 8,\n        now: datetime | None = None,\n    ) -> list[dict[str, Any]]:\n        now = now or _utcnow()\n        known_titles = {}\n        if skills:\n            for skill in skills:\n                item = skill.to_dict() if isinstance(skill, Skill) else skill\n                known_titles[item["id"]] = item\n\n        due: list[dict[str, Any]] = []\n        for skill_id, row in self.data.get("skills", {}).items():\n            next_review = _parse_iso(row.get("next_review"))\n            if next_review is None or next_review <= now:\n                due.append({\n                    "id": skill_id,\n                    "title": known_titles.get(skill_id, {}).get("title", skill_id),\n                    "mastery": row.get("mastery", 0.0),\n                    "next_review": row.get("next_review"),\n                    "attempts": row.get("attempts", 0),\n                })\n        due.sort(key=lambda item: (item["mastery"], item["next_review"] or ""))\n        return due[:limit]\n\n    def weak_skills(self, limit: int = 8) -> list[tuple[str, dict[str, Any]]]:\n        rows = list(self.data.get("skills", {}).items())\n        rows.sort(key=lambda kv: (kv[1].get("mastery", 0.0), -kv[1].get("attempts", 0)))\n        return rows[:limit]\n\n\ndef review_interval(correct: bool, confidence: int, attempts: int = 1) -> timedelta:\n    """Return the next review interval from correctness and confidence."""\n\n    if not correct:\n        return timedelta(hours=6) if attempts <= 1 else timedelta(days=1)\n    if confidence <= 2:\n        return timedelta(days=1)\n    if confidence == 3:\n        return timedelta(days=3)\n    high_confidence_days = [7, 14, 30, 60]\n    return timedelta(days=high_confidence_days[min(max(attempts - 1, 0), len(high_confidence_days) - 1)])\n\n\ndef record_attempt(\n    skill_id: str,\n    correct: bool,\n    confidence: int = 3,\n    item_type: str = "practice",\n    store: ProgressStore | None = None,\n) -> dict[str, Any]:\n    store = store or ProgressStore()\n    return store.record_attempt(skill_id, correct, confidence, item_type)\n\n\ndef check(\n    name: str,\n    actual: Any,\n    expected: Any,\n    tolerance: float | None = None,\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Friendly equality/numeric check with optional progress recording."""\n\n    if tolerance is not None and isinstance(actual, (int, float)) and isinstance(expected, (int, float)):\n        ok = math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance)\n    else:\n        ok = actual == expected\n\n    if ok:\n        print(f"PASS: {name}")\n    else:\n        print(f"TRY AGAIN: {name}")\n        print(f"  expected: {expected!r}")\n        print(f"  actual:   {actual!r}")\n\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "check", store)\n    return ok\n\n\ndef code_check(\n    name: str,\n    fn: Callable[..., Any],\n    cases: Iterable[tuple[tuple[Any, ...], Any] | tuple[tuple[Any, ...], dict[str, Any], Any]],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Run multiple function cases and report the first failure."""\n\n    all_ok = True\n    for index, case in enumerate(cases, 1):\n        if len(case) == 2:\n            args, expected = case  # type: ignore[misc]\n            kwargs = {}\n        else:\n            args, kwargs, expected = case  # type: ignore[misc]\n        try:\n            actual = fn(*args, **kwargs)\n            ok = actual == expected\n        except Exception as exc:\n            actual = f"{type(exc).__name__}: {exc}"\n            ok = False\n        if not ok:\n            all_ok = False\n            print(f"TRY AGAIN: {name} case {index}")\n            print(f"  args:     {args}")\n            print(f"  expected: {expected!r}")\n            print(f"  actual:   {actual!r}")\n            break\n    if all_ok:\n        print(f"PASS: {name} ({index} cases)")\n    if skill_id:\n        record_attempt(skill_id, all_ok, confidence, "code_check", store)\n    return all_ok\n\n\ndef normalize_answer(value: Any) -> str:\n    text = str(value).strip().lower()\n    text = re.sub(r"\\s+", " ", text)\n    text = re.sub(r"[^a-z0-9.+/=\\- ]", "", text)\n    text = re.sub(r"\\s*=\\s*", "=", text)\n    return text\n\n\ndef short_answer_check(\n    name: str,\n    answer: Any,\n    accepted: Iterable[Any],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    normalized = normalize_answer(answer)\n    accepted_norm = {normalize_answer(item) for item in accepted}\n    ok = normalized in accepted_norm\n    print(f"{\'PASS\' if ok else \'TRY AGAIN\'}: {name}")\n    if not ok:\n        print("  Re-read the prompt and try to retrieve the answer before opening a hint.")\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "short_answer", store)\n    return ok\n\n\ndef hint_box(title: str, hints: list[str], unlock: bool = False) -> None:\n    """Display staged hints. Full hints require unlock=True."""\n\n    print(title)\n    if not hints:\n        print("No hints for this item.")\n        return\n    if not unlock:\n        print(f"Hint 1: {hints[0]}")\n        print("Run again with unlock=True after a real attempt to see more.")\n        return\n    for index, hint in enumerate(hints, 1):\n        print(f"Hint {index}: {hint}")\n\n\ndef mastery_dashboard(\n    store: ProgressStore | None = None,\n    skills: Iterable[Skill | dict[str, Any]] | None = None,\n    next_notebook: str | None = None,\n) -> None:\n    store = store or ProgressStore()\n    due = store.review_queue(skills=skills)\n    weak = store.weak_skills()\n    total = len(store.data.get("skills", {}))\n    mastered = sum(1 for row in store.data.get("skills", {}).values() if row.get("mastery", 0) >= 0.8)\n\n    print("Mastery Dashboard")\n    print("=================")\n    print(f"Skills attempted: {total}")\n    print(f"Skills at mastery >= 0.80: {mastered}")\n    if due:\n        print("\\nDue for review:")\n        for item in due:\n            print(f"- {item[\'title\']} (mastery {item[\'mastery\']})")\n    else:\n        print("\\nNo due reviews yet.")\n    if weak:\n        print("\\nWeakest skills:")\n        for skill_id, row in weak:\n            print(f"- {skill_id}: mastery {row.get(\'mastery\', 0)} after {row.get(\'attempts\', 0)} attempts")\n    if next_notebook:\n        print(f"\\nRecommended next notebook: {next_notebook}")\n', encoding='utf-8')    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )progress_path = setup_colab()store = ProgressStore(progress_path)print('Ready for retrieval practice.')

In [ ]:
NOTEBOOK = {    "id": "01_introductory_coding",    "level": 1,    "title": "Algorithms 01: Introductory Coding",    "prerequisites": [        "Basic programming familiarity"    ],    "skills_taught": [        "lesson.algorithms_01_introductory_coding"    ],    "skills_practiced": [        "py.arithmetic.expressions"    ],    "next_notebook": "02_number_systems.ipynb"}SKILLS = [    {        "id": "lesson.algorithms_01_introductory_coding",        "title": "Lesson Algorithms 01 Introductory Coding",        "notebook": "01_introductory_coding.ipynb",        "level": 1    },    {        "id": "py.arithmetic.expressions",        "title": "Py Arithmetic Expressions",        "notebook": "01_introductory_coding.ipynb",        "level": 1    }]

## Before You Learn: Retrieval GateClose notes for a moment. Try to pull prerequisite ideas from memory before continuing. Getting stuck is useful data for the spaced-review system.

In [ ]:
due = store.review_queue(skills=SKILLS, limit=5)if due:    print('Due review items:')    for item in due:        print(f"- {item['title']} (mastery {item['mastery']})")else:    print('No due reviews yet. Use the recall prompt below to warm up.')

**Recall prompt.** Before reading, name one key idea or procedure you remember from: Basic programming familiarity.

In [ ]:
recall_answer = ''  # write a one-sentence memory before continuingretrieved = len(recall_answer.strip()) >= 12record = store.record_attempt('lesson.algorithms_01_introductory_coding', retrieved, confidence=3, item_type='prerequisite_recall')print('Recorded retrieval attempt.' if retrieved else 'Write a real memory first, then rerun this cell.')record

## Learning LoopUse the existing lesson cells below as the micro-lesson and practice surface. For each exercise: predict first, attempt from memory, run checks, then open explanations only after the attempt.

In [ ]:
# Google Colab Setup!pip install -q icecream line_profilerfrom icecream import icprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### The Math Academy / Eurisko Philosophy> 📖 *From Algorithms Ch. 1:* Write the following functions from scratch. Don’t use any external libraries or any built-in functions that allow you to bypass the use of for loops, while loops, or if statements in nontrivial ways. In particular, don’t use `Set` or `Counter`.In this course, we build everything from the ground up. Before you can use a library to sort an array, you must write the sorting algorithm yourself. This develops deep algorithmic intuition.### Debugging Principles1. **Print out everything.** Within the function you're debugging, print every manipulation. Bugs hide where you don't expect them.2. **Identify the first discrepancy.** Manually trace what the code *should* do step-by-step, and see where the printed output deviates from your manual trace.

### Comprehension Check ✅1. Why are we forbidden from using Python's `set()` to find the intersection of two arrays?2. What is the most effective tool for debugging when your code produces the wrong answer?<details><summary>💡 Explanation after a retrieval attempt</summary>1. Because using `set()` abstracts away the algorithmic logic of searching and comparing elements. By writing it from scratch, you learn how intersection actually works under the hood.2. Print statements! (`print()` or `ic()` from the icecream library) to trace the state of variables at every step.</details>

---## Phase 0 — Math Foundation PracticeBefore writing code, let's trace an algorithm manually.### 🎯 Your Turn: Trace `get_intersection`Suppose `array1 = [3, 1, 4]` and `array2 = [4, 5, 3]`.Trace how you would find the intersection without using `set()`.| Step | Action | Output Array ||---|---|---|| 1 | Look at `3` in array1. Is `3` in array2? Yes. | `[3]` || 2 | Look at `1` in array1. Is `1` in array2? No. | `[3]` || 3 | Look at `4` in array1. Is `4` in array2? Yes. | `[3, 4]` |Write a simple loop to verify this trace computationally.

In [ ]:
def simple_intersection(a1, a2):    out = []    # YOUR CODE HERE: use a for loop to check elements of a1 against a2    pass# assert simple_intersection([3, 1, 4], [4, 5, 3]) == [3, 4]# print('Phase 0 passed ✅')

---## Phase 1 — Micro-Scaffolded Algorithm ConstructionNow we will implement the 6 core exercises from Chapter 1. Remember: **No imports, no `set()`, no `Counter()`.**

### Exercise 1: `check_if_symmetric(string)`Return `True` if a string is a palindrome, `False` otherwise. Ignore capitalization.

In [ ]:
def check_if_symmetric(string):    '''Return True if string is a palindrome, ignoring case.'''    # YOUR CODE HERE    pass# Tests:# assert check_if_symmetric('racecar') == True# assert check_if_symmetric('Racecar') == True# assert check_if_symmetric('hello') == False

### Exercise 2: `convert_to_numbers(string)`Return an array of numbers corresponding to letters: space=0, a=1, b=2, etc.

In [ ]:
def convert_to_numbers(string):    '''Convert string to an array of integers.'''    # YOUR CODE HERE    pass# Test:# assert convert_to_numbers('a cat') == [1, 0, 3, 1, 20]

### Exercise 3: `get_intersection(array1, array2)`Return elements in both arrays, with NO duplicates in the output.

In [ ]:
def get_intersection(array1, array2):    '''Return intersection without duplicates.'''    # YOUR CODE HERE    pass# Test:# assert sorted(get_intersection([1, 2, 2, 3], [2, 3, 4])) == [2, 3]

### Exercise 4: `get_union(array1, array2)`Return elements in either array, with NO duplicates.

In [ ]:
def get_union(array1, array2):    '''Return union without duplicates.'''    # YOUR CODE HERE    pass# Test:# assert sorted(get_union([1, 2], [2, 3])) == [1, 2, 3]

### Exercise 5: `count_characters(string)`Return a dictionary counting occurrences of each character (case-insensitive).

In [ ]:
def count_characters(string):    '''Return a dict of char counts.'''    # YOUR CODE HERE    pass# Test:# assert count_characters('A cat') == {'a': 2, ' ': 1, 'c': 1, 't': 1}

### Exercise 6: `is_prime(N)`Check if N > 1 is prime by testing divisors up to floor(N/2).

In [ ]:
def is_prime(N):    '''Return True if N is prime, else False.'''    if N <= 1: return False    # YOUR CODE HERE    pass# Tests:# assert is_prime(2) == True# assert is_prime(4) == False# assert is_prime(17) == True

---## Phase 2 — Experimental Verification### Pathological Case: Time Complexity of `is_prime`What happens if we test a huge prime number? Let's use `%timeit` to see how long our algorithm takes.

In [ ]:
# Try to check a very large prime number: 15485863 (the 1 millionth prime)import timestart = time.time()# is_prime(15485863) # uncomment to testprint(f"Time taken: {time.time() - start:.4f} seconds")

### 🔍 Reflection1. If we use the condition `n <= N/2`, we check $\approx 7.5$ million numbers.2. How could we optimize this bound? (Hint: Think about factors appearing in pairs).<details><summary>💡 Explanation after a retrieval attempt</summary>Factors come in pairs. If $A \times B = N$, one of them must be $\leq \sqrt{N}$.So we only need to check up to $\lfloor \sqrt{N} \rfloor$. For $N=15485863$, $\sqrt{N} \approx 3935$. This reduces the loop from 7.5 million iterations to 4 thousand!</details>

---## Phase 3 — Olympiad Extension### Challenge: The Sieve of EratosthenesIf you need to find *all* prime numbers up to $M$, calling `is_prime()` $M$ times is too slow.The Sieve of Eratosthenes finds all primes up to $M$ efficiently. Implement it from scratch.

In [ ]:
def sieve_of_eratosthenes(M):    '''Return a list of all primes up to M.'''    # Create a boolean array [True] * (M+1)    # YOUR CODE HERE    pass# Test:# assert sieve_of_eratosthenes(30) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]

### Chalkboard DefenseExplain why `get_intersection` takes $O(N^2)$ time if implemented with nested loops (or `in` on lists), and how you could make it $O(N \log N)$ using sorting, or $O(N)$ using dictionaries (which are allowed as long as you don't use `set()`).*(Write your defense below)*<details><summary>💡 Sketch after a retrieval attempt</summary>If we loop over `array1` (length $N$) and for each element, scan `array2` (length $N$) using `in`, the total operations are $N \times N = N^2$.If we use a dictionary to store elements of `array2` first ($O(N)$), dictionary lookups are $O(1)$. So looping through `array1` and checking the dictionary takes $O(N)$. Total time: $O(N)$.</details>---📚 **Next:** Algorithms 02 (Computational Math & Base Conversions)

## End-of-Notebook ReviewRetrieve the core idea one more time before leaving. This final recall makes the next review easier.

In [ ]:
final_recall = ''  # summarize the most important idea in your own wordsconfidence = 3  # set 1-5 after answeringok = len(final_recall.strip()) >= 20store.record_attempt('lesson.algorithms_01_introductory_coding', ok, confidence=confidence, item_type='end_review')mastery_dashboard(store=store, skills=SKILLS, next_notebook='02_number_systems.ipynb')